# Kaggle: EagleVision Phase 1 Eval and Inference

Use this notebook after a proper Kaggle training run, or when you want evaluation-only workflows.

It covers:

- cloning and installing the repo in the session
- normalizing an attached ScanNet-style dataset
- selecting a trained checkpoint
- running baseline and adapted evaluation
- comparing metrics
- running depth inference and a geometric render demo


In [ ]:
def run(cmd: list[str], cwd: Path | None = None, capture: bool = False):
    print('$', ' '.join(cmd))
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        check=True,
        text=True,
        capture_output=capture,
    )
    if capture:
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr)
        return completed
    return None

if REPO_DIR.exists():
    run(['git', 'fetch', '--all'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
else:
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])

run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], cwd=REPO_DIR)
print('repo ready:', REPO_DIR)


## Point To Your Attached Dataset And Optional Training Outputs

Recommended Kaggle dataset for raw scenes:

- `klein2111/scannet-2d`
- Kaggle link: `https://www.kaggle.com/datasets/klein2111/scannet-2d`
- default path: `/kaggle/input/scannet-2d`

Optional second input:

- previous training outputs exported as a Kaggle dataset: `/kaggle/input/eaglevision-phase1-run`


In [ ]:
RAW_DATASET_DIR = KAGGLE_INPUT / 'scannet-2d'
TRAIN_RUN_INPUT_DIR = KAGGLE_INPUT / 'eaglevision-phase1-run'
RAW_DATASET_DIR, TRAIN_RUN_INPUT_DIR

In [ ]:
import zipfile

TARGET_DATA_ROOT = REPO_DIR / 'data' / 'scannet'
TARGET_DATA_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACT_ROOT = KAGGLE_WORKING / 'scannet_extracted'
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

def discover_scene_dirs(raw_root: Path) -> list[Path]:
    scenes = []
    for candidate in sorted(raw_root.rglob('*')):
        if candidate.is_dir() and (candidate / 'color').exists() and (candidate / 'depth').exists() and (candidate / 'pose').exists():
            scenes.append(candidate)
    deduped = []
    seen = set()
    for scene in scenes:
        if scene.name not in seen:
            deduped.append(scene)
            seen.add(scene.name)
    return deduped

def extract_archives_if_needed(raw_root: Path, extract_root: Path) -> None:
    archives = sorted(list(raw_root.glob('*.zip')) + list(raw_root.glob('*.tar.gz')) + list(raw_root.glob('*.tgz')))
    for archive in archives:
        target_dir = extract_root / archive.stem.replace('.tar', '')
        if target_dir.exists():
            continue
        target_dir.mkdir(parents=True, exist_ok=True)
        if archive.suffix == '.zip':
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(target_dir)
        else:
            shutil.unpack_archive(str(archive), str(target_dir))

def materialize_scene(scene_dir: Path, target_root: Path) -> None:
    dst = target_root / scene_dir.name
    if dst.exists():
        return
    try:
        os.symlink(scene_dir, dst, target_is_directory=True)
    except OSError:
        shutil.copytree(scene_dir, dst)

extract_archives_if_needed(RAW_DATASET_DIR, EXTRACT_ROOT)
scene_dirs = discover_scene_dirs(RAW_DATASET_DIR)
if not scene_dirs:
    scene_dirs = discover_scene_dirs(EXTRACT_ROOT)
if not scene_dirs:
    raise RuntimeError('No ScanNet-style scenes found in the attached dataset or extracted archives.')
for scene_dir in scene_dirs:
    materialize_scene(scene_dir, TARGET_DATA_ROOT)

scene_ids = sorted([path.name for path in TARGET_DATA_ROOT.iterdir() if path.is_dir()])
split_index = max(1, int(0.8 * len(scene_ids)))
train_scenes = scene_ids[:split_index]
val_scenes = scene_ids[split_index:] if split_index < len(scene_ids) else scene_ids[:1]
len(scene_ids), val_scenes[:10]

## Ensure The Baseline Checkpoint Exists

In [ ]:
run(
    [
        'python', '-m', 'baseline.depth_anything_v2', 'download',
        '--mode', 'metric',
        '--profile', 'hypersim',
        '--encoder', 'vits',
    ],
    cwd=REPO_DIR,
)

## Build An Evaluation Config

In [ ]:
CONFIG_DIR = REPO_DIR / 'outputs' / 'kaggle_configs'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

eval_config = {
    'device': 'cuda' if shutil.which('nvidia-smi') else 'cpu',
    'data': {
        'root': str(TARGET_DATA_ROOT),
        'image_size': [240, 320],
        'intrinsics': [
            [577.8706, 0.0, 159.5],
            [0.0, 577.8706, 119.5],
            [0.0, 0.0, 1.0],
        ],
        'pairing': {
            'min_translation_m': 0.02,
            'max_translation_m': 0.30,
            'min_rotation_deg': 1.0,
            'max_rotation_deg': 10.0,
            'max_index_gap': 30,
        },
        'splits': {
            'val': {'scenes': val_scenes},
        },
    },
    'model': {
        'depth': {
            'mode': 'metric',
            'encoder': 'vits',
            'profile': 'hypersim',
            'checkpoint_path': None,
            'freeze_backbone': True,
            'adapter_hidden_channels': 32,
        }
    },
    'eval': {'batch_size': 2},
    'losses': {
        'weights': {
            'target_rgb': 1.0,
            'cycle_rgb': 1.0,
            'cycle_depth': 0.5,
            'target_depth': 0.25,
        }
    },
}

eval_config_path = CONFIG_DIR / 'eval_phase1_kaggle.yaml'
with eval_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(eval_config, handle, sort_keys=False)
eval_config_path

## Select A Trained Checkpoint

Point this at a checkpoint produced by the training notebook. If you exported a Kaggle dataset from a previous run, update the path below.

In [ ]:
CHECKPOINT_PATH = TRAIN_RUN_INPUT_DIR / 'checkpoints' / 'step_0000100.pt'
CHECKPOINT_PATH

## Run Baseline And Adapted Evaluation

In [ ]:
baseline_path = REPO_DIR / 'outputs' / 'baseline_metrics.yaml'
adapted_path = REPO_DIR / 'outputs' / 'adapted_metrics.yaml'
with baseline_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(baseline_metrics, handle, sort_keys=False)
with adapted_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(adapted_metrics, handle, sort_keys=False)
baseline_path, adapted_path


In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f'Checkpoint not found: {CHECKPOINT_PATH}. Edit CHECKPOINT_PATH to match your Kaggle input artifact.'
    )

adapted_eval = run(
    ['python', '-m', 'eaglevision.cli.eval', '--config', str(eval_config_path), '--checkpoint', str(CHECKPOINT_PATH)],
    cwd=REPO_DIR,
    capture=True,
)

adapted_metrics = {}
for line in adapted_eval.stdout.splitlines():
    if ': ' not in line:
        continue
    key, value = line.split(': ', 1)
    try:
        adapted_metrics[key.strip()] = float(value.strip())
    except ValueError:
        pass

adapted_metrics


## Optional: Save Metric YAMLs For `compare_models`

The current CLI prints to stdout. If you want direct notebook-side comparison artifacts, create small YAML files manually.

## Evaluation Summary Table

Create compact summary artifacts for submission and reviewer inspection.


In [ ]:
import csv

rows = []
all_metric_names = sorted(set(baseline_metrics) | set(adapted_metrics))
for metric_name in all_metric_names:
    baseline_value = baseline_metrics.get(metric_name)
    adapted_value = adapted_metrics.get(metric_name)
    delta = None
    if baseline_value is not None and adapted_value is not None:
        delta = adapted_value - baseline_value
    rows.append({
        'metric': metric_name,
        'baseline': baseline_value,
        'adapted': adapted_value,
        'delta': delta,
    })

summary_dir = REPO_DIR / 'outputs' / 'eval_submission_summary'
summary_dir.mkdir(parents=True, exist_ok=True)
summary_json_path = summary_dir / 'comparison_metrics.json'
summary_csv_path = summary_dir / 'comparison_metrics.csv'
with summary_json_path.open('w', encoding='utf-8') as handle:
    json.dump(rows, handle, indent=2)
with summary_csv_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['metric', 'baseline', 'adapted', 'delta'])
    writer.writeheader()
    writer.writerows(rows)

rows


## Package Evaluation Artifacts

This creates a zip archive containing the metric YAMLs, comparison files, and inference outputs.


In [ ]:
export_root = REPO_DIR / 'outputs' / 'eval_submission_bundle'
export_root.mkdir(parents=True, exist_ok=True)
for artifact in [baseline_path, adapted_path, summary_json_path, summary_csv_path, infer_output]:
    if Path(artifact).exists():
        shutil.copy2(artifact, export_root / Path(artifact).name)
archive_base = REPO_DIR / 'outputs' / 'eval_submission_bundle'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=export_root)
archive_path


In [ ]:
run(
    [
        'python', '-m', 'eaglevision.cli.compare_models',
        '--baseline-metrics', str(baseline_path),
        '--adapted-metrics', str(adapted_path),
    ],
    cwd=REPO_DIR,
)

## Run Inference On One Attached Image

In [ ]:
first_scene = scene_ids[0]
sample_image = next((TARGET_DATA_ROOT / first_scene / 'color').glob('*.jpg'))
infer_output = REPO_DIR / 'outputs' / 'kaggle_infer_depth.png'

run(
    [
        'python', '-m', 'eaglevision.cli.infer_depth',
        '--input', str(sample_image),
        '--output', str(infer_output),
        '--mode', 'metric',
        '--encoder', 'vits',
        '--profile', 'hypersim',
    ],
    cwd=REPO_DIR,
)
infer_output

## Optional: Geometric Novel-View Render Demo

This requires a real depth `.npy`, intrinsics text file, and relative transform text file. If your dataset stores depth only as `.png`, convert it first.

In [ ]:
print('Prepare depth.npy, intrinsics.txt, and transform.txt if you want to run render_novel_view here.')